<a href="https://colab.research.google.com/github/Akshitha94127/NASSCOM/blob/main/U8_Data_Cleaning_Part1_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.3


In [ ]:
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})
raw

,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,NaN,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,N/A,2024-01-08
4,5,Eve,-1.0,Mumbai,150.0,2024-01-09
5,6,Fin,38.0,bombay,99000,2024-01-10
6,7,Gus,29.0,Pune.,110.0,2024-01-11
7,7,Gus,29.0,Pune.,110.0,2024-01-11


In [ ]:
#1A. A FEW COMMANDS REVEAL MOST PROBLEMS
df = raw.copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      8 non-null      int64  
 1   name    8 non-null      object 
 2   age     7 non-null      float64
 3   city    8 non-null      object 
 4   spend   8 non-null      object 
 5   date    8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [ ]:
#1B. MISSING COUNTS, DUPLICATES & RANGES
print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")

Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 1

Note: spend is type object -> stored as text!


In [ ]:
#LAB EXERCISE 1 — Profile it yourself
import pandas as pd

df_name=[name for name,obj in globals().items() if isinstance(obj,pd.DataFrame)][0]
df=globals()[df_name]

print(df.duplicated().sum())
print(df.isnull().sum())

# Problems I can see: duplicate rows, missing values, inconsistent data


1
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64


In [ ]:
#2A. UNMASK DISGUISED MISSING VALUES
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')  # 'N/A' -> NaN, text -> number
df['age']   = df['age'].replace(-1, np.nan)                # sentinel -> NaN

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())


Missing after unmasking:
age      2
spend    1
dtype: int64


In [ ]:
#2B. HANDLE THE GAPS (impute)
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())

Missing after imputing: 0


In [ ]:
#LAB EXERCISE 2 — Compare drop vs impute
import pandas as pd

ex=raw.copy()

ex['spend']=pd.to_numeric(ex['spend'],errors='coerce')
ex['age']=ex['age'].replace(-1,pd.NA)

drop_version=ex.dropna()
impute_version=ex.copy()

impute_version['age']=impute_version['age'].fillna(impute_version['age'].median())
impute_version['spend']=impute_version['spend'].fillna(impute_version['spend'].median())

print('Original rows:',len(ex))
print('After dropna:',len(drop_version))
print('After imputation:',len(impute_version))
print('Rows lost by dropping:',len(ex)-len(drop_version))


Original rows: 8
After dropna: 8
After imputation: 8
Rows lost by dropping: 0


In [ ]:
#3A. DROP DUPLICATE ROWS
print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')


Before: (8, 6)
After : (7, 6) -> removed the repeated Gus row


In [ ]:
#3B. FIX DATA TYPES
df['date'] = pd.to_datetime(df['date'])
df['city'] = df['city'].astype('string')
print(df.dtypes)


id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


/tmp/ipykernel_908/3607585212.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/tmp/ipykernel_908/3607585212.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['city'] = df['city'].astype('string')


In [ ]:
#LAB EXERCISE 3 — Dedupe & retype
import pandas as pd

raw=pd.DataFrame({
    'spend':['100','200','150','100'],
    'date':['2024-01-01','2024-01-02','2024-01-03','2024-01-01']
})

ex=raw.copy()

ex['spend']=pd.to_numeric(ex['spend'],errors='coerce')
ex['date']=pd.to_datetime(ex['date'],errors='coerce')

ex=ex.drop_duplicates()

print(ex.dtypes)
print(ex.shape)


spend             int64
date     datetime64[ns]
dtype: object
(3, 2)


In [ ]:
#4A. THE IQR RULE
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])

Q1=115.2  Q3=175.1  IQR=59.8
Normal range: 25.5 to 264.9

Outlier rows:
  name    spend
5  Fin  99000.0


In [ ]:
#4B. ONE WAY TO TREAT THEM — CAP (winsorise)
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])


  name    spend  spend_capped
0  Ana    120.5       120.500
1   Bo     80.0        80.000
2   Cy    200.2       200.200
3   Di    120.5       120.500
4  Eve    150.0       150.000
5  Fin  99000.0       264.875
6  Gus    110.0       110.000


/tmp/ipykernel_908/1748089870.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['spend_capped'] = df['spend'].clip(lower=low, upper=high)


In [ ]:
#LAB EXERCISE 4 — Find the outliers
import pandas as pd

df=pd.DataFrame({'age':[18,22,25,30,35,40,45,50,55,60]})

Q1=df['age'].quantile(0.25)
Q3=df['age'].quantile(0.75)
IQR=Q3-Q1

lower=Q1-1.5*IQR
upper=Q3+1.5*IQR

print('Q1:',Q1)
print('Q3:',Q3)
print('IQR:',IQR)
print('Lower:',lower)
print('Upper:',upper)

print(df[(df['age']<lower)|(df['age']>upper)])


Q1: 26.25
Q3: 48.75
IQR: 22.5
Lower: -7.5
Upper: 82.5
Empty DataFrame
Columns: [age]
Index: []


In [4]:
#5A. THE PROBLEM — ONE CITY, MANY SPELLINGS
import pandas as pd
df = pd.DataFrame({
    'city': [' Pune ', 'pune', 'Pune.', 'Mumbai']
})
print(df['city'].value_counts())

city
 Pune     1
pune      1
Pune.     1
Mumbai    1
Name: count, dtype: int64


In [5]:
#5B. STANDARDISE THE STRINGS
s = df['city'].astype('string')
s = s.str.strip()
s = s.str.lower()
s = s.str.replace('.', '', regex=False)
s = s.replace({'bombay': 'mumbai'})
df['city'] = s
print(df['city'].value_counts())

city
pune      3
mumbai    1
Name: count, dtype: Int64


In [6]:
#LAB EXERCISE 5 — Clean a messy column
import pandas as pd
messy = pd.Series([' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
                 dtype='string')
# 1.
messy = messy.str.strip().str.lower()
# 2.
messy = messy.replace('n.y.', 'new york')
# 3.
print(messy.value_counts())

london      3
new york    3
Name: count, dtype: Int64


In [11]:
clean = df.drop(columns=['spend_capped'], errors='ignore')
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))

Final shape: (4, 1)
Missing values: 0
Duplicates    : 2
